# 03 - Model Training, Hyperparameter Tuning & Experiment Tracking

## Anomaly Detection for Reconciliation Variances

This notebook covers:
- Multiple anomaly detection algorithms (Isolation Forest, One-Class SVM, LOF)
- Hyperparameter tuning with cross-validation
- Model comparison and selection
- Experiment tracking with Snowflake ML

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from snowflake.snowpark.context import get_active_session
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

session = get_active_session()

FEATURE_COLUMNS = [
    'VARIANCE_AMOUNT', 'GL_BANK_DIFF', 'GL_SUBLEDGER_DIFF', 'RECON_COUNT', 'UNIDENTIFIED_AMOUNT',
    'VARIANCE_PCT_CHANGE', 'VARIANCE_Z_SCORE', 'VARIANCE_Z_SCORE_PERIOD', 'VARIANCE_PCT_OF_BALANCE',
    'VARIANCE_VS_ROLLING_MAX', 'IS_ABOVE_P95', 'GL_BANK_DIFF_RATIO', 'BALANCE_CHANGE_PCT',
    'HIERARCHY_DEPTH_NORMALIZED', 'IS_KEY_ACCOUNT_FLAG', 'ROLLING_AVG_VARIANCE_3', 'ROLLING_STD_VARIANCE_3',
    'ROLLING_MAX_VARIANCE_6', 'ENTITY_AVG_VARIANCE', 'ENTITY_ASSIGNMENT_COUNT'
]

print(f"Connected to Snowflake")

## 1. Load Training Artifacts

In [ ]:
try:
    with open('training_artifacts.pkl', 'rb') as f:
        artifacts = pickle.load(f)
except FileNotFoundError:
    raise FileNotFoundError("Run 02_feature_engineering.ipynb first to generate training_artifacts.pkl")

X_train_scaled = artifacts['X_train_scaled']
y_train = artifacts['y_train']
X_val_scaled = artifacts['X_val_scaled']
y_val = artifacts['y_val']
scaler = artifacts['scaler']
feature_columns = artifacts['feature_columns']

print(f"Training set: {X_train_scaled.shape[0]:,} samples")
print(f"Validation set: {X_val_scaled.shape[0]:,} samples")
print(f"Features: {len(feature_columns)}")
print(f"Training anomaly rate: {100*y_train.mean():.2f}%")
print(f"Validation anomaly rate: {100*y_val.mean():.2f}%")

## 2. Initialize Training Pipeline

In [ ]:
def evaluate_model(y_true, y_pred, y_scores=None):
    """Calculate evaluation metrics."""
    y_pred_binary = (y_pred == -1).astype(int)
    metrics = {
        'precision': precision_score(y_true, y_pred_binary, zero_division=0),
        'recall': recall_score(y_true, y_pred_binary, zero_division=0),
        'f1_score': f1_score(y_true, y_pred_binary, zero_division=0),
    }
    if y_scores is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, -y_scores)
        except:
            metrics['roc_auc'] = 0.0
    return metrics

results = {}
best_models = {}

print("Available model types:")
print("  - IsolationForest")
print("  - OneClassSVM")
print("  - LocalOutlierFactor")

## 3. Train Isolation Forest

In [ ]:
print("Training Isolation Forest with hyperparameter tuning...")
start = time.time()

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_samples': ['auto', 0.5],
    'contamination': [0.05, 0.10, 0.15],
    'random_state': [42]
}

if_results = []
best_if_f1 = 0
best_if_model = None
best_if_params = None

tscv = TimeSeriesSplit(n_splits=3)

for i, params in enumerate(ParameterGrid(param_grid)):
    if i >= 15:
        break
    
    cv_scores = []
    for train_idx, test_idx in tscv.split(X_train_scaled):
        X_cv_train, X_cv_test = X_train_scaled[train_idx], X_train_scaled[test_idx]
        y_cv_train, y_cv_test = y_train.iloc[train_idx], y_train.iloc[test_idx]
        
        model = IsolationForest(**params)
        model.fit(X_cv_train)
        y_pred = model.predict(X_cv_test)
        y_scores = model.decision_function(X_cv_test)
        
        metrics = evaluate_model(y_cv_test, y_pred, y_scores)
        cv_scores.append(metrics['f1_score'])
    
    final_model = IsolationForest(**params)
    final_model.fit(X_train_scaled)
    y_val_pred = final_model.predict(X_val_scaled)
    y_val_scores = final_model.decision_function(X_val_scaled)
    val_metrics = evaluate_model(y_val, y_val_pred, y_val_scores)
    
    if_results.append({
        'model_type': 'IsolationForest',
        'params': params,
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': time.time() - start
    })
    
    if val_metrics['f1_score'] > best_if_f1:
        best_if_f1 = val_metrics['f1_score']
        best_if_model = final_model
        best_if_params = params

results['IsolationForest'] = if_results
best_models['IsolationForest'] = (best_if_model, best_if_params, best_if_f1)

print(f"\nCompleted in {time.time()-start:.1f}s")
print(f"Best Isolation Forest F1: {best_if_f1:.4f}")
print(f"Best params: {best_if_params}")

## 4. Train One-Class SVM

In [ ]:
print("Training One-Class SVM with hyperparameter tuning...")
start = time.time()

param_grid = {
    'kernel': ['rbf', 'poly'],
    'nu': [0.05, 0.10, 0.15],
    'gamma': ['scale', 'auto']
}

max_samples = min(50000, len(X_train_scaled))
X_train_subset = X_train_scaled[:max_samples]
y_train_subset = y_train.iloc[:max_samples]
X_val_subset = X_val_scaled[:20000] if len(X_val_scaled) > 20000 else X_val_scaled
y_val_subset = y_val.iloc[:20000] if len(y_val) > 20000 else y_val

svm_results = []
best_svm_f1 = 0
best_svm_model = None
best_svm_params = None

for i, params in enumerate(ParameterGrid(param_grid)):
    if i >= 9:
        break
    
    model = OneClassSVM(**params)
    model.fit(X_train_subset)
    y_val_pred = model.predict(X_val_subset)
    y_val_scores = model.decision_function(X_val_subset)
    val_metrics = evaluate_model(y_val_subset, y_val_pred, y_val_scores)
    
    svm_results.append({
        'model_type': 'OneClassSVM',
        'params': params,
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': time.time() - start
    })
    
    if val_metrics['f1_score'] > best_svm_f1:
        best_svm_f1 = val_metrics['f1_score']
        best_svm_model = model
        best_svm_params = params

results['OneClassSVM'] = svm_results
best_models['OneClassSVM'] = (best_svm_model, best_svm_params, best_svm_f1)

print(f"\nCompleted in {time.time()-start:.1f}s")
print(f"Best One-Class SVM F1: {best_svm_f1:.4f}")
print(f"Best params: {best_svm_params}")

## 5. Train Local Outlier Factor

In [ ]:
print("Training Local Outlier Factor with hyperparameter tuning...")
start = time.time()

param_grid = {
    'n_neighbors': [10, 20, 50],
    'contamination': [0.05, 0.10, 0.15],
    'metric': ['euclidean', 'manhattan']
}

lof_results = []
best_lof_f1 = 0
best_lof_model = None
best_lof_params = None

for i, params in enumerate(ParameterGrid(param_grid)):
    if i >= 9:
        break
    
    model = LocalOutlierFactor(**params, novelty=True)
    model.fit(X_train_subset)
    y_val_pred = model.predict(X_val_subset)
    y_val_scores = model.decision_function(X_val_subset)
    val_metrics = evaluate_model(y_val_subset, y_val_pred, y_val_scores)
    
    lof_results.append({
        'model_type': 'LocalOutlierFactor',
        'params': params,
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': time.time() - start
    })
    
    if val_metrics['f1_score'] > best_lof_f1:
        best_lof_f1 = val_metrics['f1_score']
        best_lof_model = model
        best_lof_params = params

results['LocalOutlierFactor'] = lof_results
best_models['LocalOutlierFactor'] = (best_lof_model, best_lof_params, best_lof_f1)

print(f"\nCompleted in {time.time()-start:.1f}s")
print(f"Best LOF F1: {best_lof_f1:.4f}")
print(f"Best params: {best_lof_params}")

## 6. Model Comparison

In [ ]:
all_results = []
for model_type, model_results in results.items():
    all_results.extend(model_results)
results_df = pd.DataFrame(all_results)

print(f"Total configurations evaluated: {len(results_df)}")

best_per_model = results_df.loc[results_df.groupby('model_type')['val_f1'].idxmax()]
print("\nBest configuration per model type:")
best_per_model[['model_type', 'val_precision', 'val_recall', 'val_f1', 'val_roc_auc', 'train_time']]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
for model_type in results_df['model_type'].unique():
    model_data = results_df[results_df['model_type'] == model_type]
    ax1.scatter(model_data['val_precision'], model_data['val_recall'], label=model_type, s=100, alpha=0.7)
ax1.set_xlabel('Precision')
ax1.set_ylabel('Recall')
ax1.set_title('Precision vs Recall by Model Type')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
best_f1_by_model = results_df.loc[results_df.groupby('model_type')['val_f1'].idxmax()]
colors = ['steelblue', 'coral', 'green'][:len(best_f1_by_model)]
ax2.bar(best_f1_by_model['model_type'], best_f1_by_model['val_f1'], color=colors)
ax2.set_xlabel('Model Type')
ax2.set_ylabel('Best F1 Score')
ax2.set_title('Best F1 Score by Model Type')
ax2.tick_params(axis='x', rotation=45)

ax3 = axes[1, 0]
ax3.boxplot([results_df[results_df['model_type'] == m]['val_f1'] for m in results_df['model_type'].unique()],
           labels=results_df['model_type'].unique())
ax3.set_ylabel('F1 Score')
ax3.set_title('F1 Score Distribution by Model')
ax3.tick_params(axis='x', rotation=45)

ax4 = axes[1, 1]
metrics = ['val_precision', 'val_recall', 'val_f1', 'val_roc_auc']
x = np.arange(len(metrics))
width = 0.25
for i, model_type in enumerate(best_f1_by_model['model_type'].unique()):
    model_row = best_f1_by_model[best_f1_by_model['model_type'] == model_type].iloc[0]
    values = [model_row[m] for m in metrics]
    ax4.bar(x + i*width, values, width, label=model_type)
ax4.set_xlabel('Metric')
ax4.set_ylabel('Score')
ax4.set_title('Best Model Metrics Comparison')
ax4.set_xticks(x + width)
ax4.set_xticklabels(['Precision', 'Recall', 'F1', 'ROC-AUC'])
ax4.legend()

plt.tight_layout()
plt.show()

## 7. Best Overall Model

In [ ]:
best_f1 = 0
best_model = None
best_name = None
best_params = None

for model_name, (model, params, f1) in best_models.items():
    if model is not None and f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = model_name
        best_params = params

print(f"\n{'='*60}")
print(f"BEST OVERALL MODEL: {best_name}")
print(f"{'='*60}")
print(f"F1 Score: {best_f1:.4f}")
print(f"Parameters: {best_params}")

## 8. Score Distribution Analysis

In [ ]:
if best_model is not None:
    y_scores = best_model.decision_function(X_val_scaled)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(y_scores[y_val == 0], bins=50, alpha=0.7, label='Normal', density=True)
    ax.hist(y_scores[y_val == 1], bins=50, alpha=0.7, label='Anomaly', density=True)
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.set_title(f'Score Distribution: Normal vs Anomaly ({best_name})')
    ax.legend()
    plt.show()
else:
    print("No model trained successfully")

## 9. Save Training Results

In [ ]:
training_results = {
    'best_models': best_models,
    'all_results': results_df,
    'best_overall': (best_model, best_name, best_params, best_f1),
    'X_train_scaled': X_train_scaled,
    'X_val_scaled': X_val_scaled,
    'y_train': y_train,
    'y_val': y_val,
    'scaler': scaler,
    'feature_columns': feature_columns
}

with open('training_results.pkl', 'wb') as f:
    pickle.dump(training_results, f)

print("Training results saved to training_results.pkl")
print("\nModel Training Complete!")
print("Proceed to 04_deployment_registry.ipynb")